# EfficientNet-B0 

36×256 tiles → 6×6 montage,
inverted, ordinal-BCE, 5 outputs on the pre-built tile cache, so the result is
reproducible.



In [ ]:
DEBUG = False   # True = 200 slides / 2 epochs to test the loop

In [ ]:
!pip install -q timm

In [ ]:
import os, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import timm, albumentations
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import clear_output
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| timm', timm.__version__)

In [ ]:
CACHE_DIR   = '/kaggle/input/datasets/shashaboii/panda-tiles-36x256/tiles_cache'   # shared cache dataset
OUT_DIR     = '/kaggle/working'
MODEL_NAME  = 'effnetb0_36x256_fold0'
BACKBONE    = 'efficientnet_b0'
fold        = 0

# To CONTINUE a previous run: publish its output, attach it, and set this to the
# '..._last.pth' file (e.g. '../input/effnetb0-ckpt/effnetb0_36x256_fold0_last.pth').
CKPT_RESUME = None

image_size = 256; n_tiles = 36
batch_size = 4; num_workers = 4
init_lr = 3e-4; warmup_factor = 10; warmup_epo = 1
n_epochs = 2 if DEBUG else 20
USE_AMP = True

try:
    df = pd.read_csv(os.path.join(CACHE_DIR, 'train.csv'))
except Exception:
    df = pd.read_csv('/kaggle/input/competitions/prostate-cancer-grade-assessment')
have = set(f[:-4] for f in os.listdir(CACHE_DIR) if f.endswith('.jpg'))
df = df[df.image_id.isin(have)].reset_index(drop=True)
if DEBUG: df = df.sample(200, random_state=42).reset_index(drop=True)
print('cached montages:', len(have), '| training rows:', len(df))

In [ ]:
transforms_train = albumentations.Compose([
    albumentations.Transpose(p=0.5),
    albumentations.VerticalFlip(p=0.5),
    albumentations.HorizontalFlip(p=0.5)])
transforms_val = albumentations.Compose([])

class CachedDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True); self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        m = np.array(Image.open(os.path.join(CACHE_DIR, f'{row.image_id}.jpg')))
        m = 255 - m                                   # invert, exactly like the public model expects
        if self.transform is not None:
            m = self.transform(image=m)['image']
        m = (m.astype(np.float32) / 255).transpose(2, 0, 1)
        label = np.zeros(5, np.float32); label[:int(row.isup_grade)] = 1.
        return torch.tensor(m), torch.tensor(label)

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=42)
df['fold'] = -1
for i, (_, va) in enumerate(skf.split(df, df.isup_grade)):
    df.loc[va, 'fold'] = i
df_this  = df[df.fold != fold]
df_valid = df[df.fold == fold].reset_index(drop=True)
print('train', len(df_this), '| val', len(df_valid), flush=True)

In [ ]:
class enetv2(nn.Module):
    def __init__(self, backbone=BACKBONE, out_dim=5, pretrained=True):
        super().__init__()
        self.enet = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.myfc = nn.Linear(self.enet.num_features, out_dim)
    def forward(self, x):
        return self.myfc(self.enet(x))

criterion = nn.BCEWithLogitsLoss()
def safe_qwk(p, t):
    return cohen_kappa_score(p, t, weights='quadratic') if len(p) else float('nan')

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def train_epoch(loader, optimizer, epoch):
    model.train(); losses = []
    pbar = tqdm(loader, desc=f'epoch {epoch}/{n_epochs}')
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            loss = criterion(model(data), target)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        losses.append(loss.item()); pbar.set_postfix(loss=float(np.mean(losses[-20:])))
    return float(np.mean(losses))

def val_epoch(loader, df_valid, get_output=False):
    model.eval(); L, P, T = [], [], []
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            logits = model(data); L.append(criterion(logits, target).item())
            P.append(logits.sigmoid().sum(1).round()); T.append(target.sum(1))
    P = torch.cat(P).cpu().numpy(); T = torch.cat(T).cpu().numpy()
    if get_output: return P, T
    mk = (df_valid.data_provider == 'karolinska').values
    mr = (df_valid.data_provider == 'radboud').values
    return (float(np.mean(L)), (P == T).mean()*100, safe_qwk(P, T),
            safe_qwk(P[mk], df_valid.isup_grade.values[mk]),
            safe_qwk(P[mr], df_valid.isup_grade.values[mr]))

def live(h):
    clear_output(wait=True); ep = range(1, len(h['tl'])+1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(ep, h['tl'], '-o', label='train'); ax[0].plot(ep, h['vl'], '-o', label='val')
    ax[0].set_title('BCE loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
    ax[1].plot(ep, h['qwk'], '-o', label='QWK'); ax[1].plot(ep, h['qk'], '--', label='Karolinska')
    ax[1].plot(ep, h['qr'], '--', label='Radboud'); ax[1].set_ylim(0,1)
    ax[1].set_title('QWK'); ax[1].set_xlabel('epoch'); ax[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
train_loader = DataLoader(CachedDataset(df_this,  transforms_train),
                          batch_size=batch_size, shuffle=True,  num_workers=num_workers)
valid_loader = DataLoader(CachedDataset(df_valid, transforms_val),
                          batch_size=batch_size, shuffle=False, num_workers=num_workers)

model = enetv2(pretrained=True).to(device)
optimizer = optim.Adam(model.parameters(), lr=init_lr)
warmup = LinearLR(optimizer, start_factor=1/warmup_factor, total_iters=warmup_epo)
cosine = CosineAnnealingLR(optimizer, T_max=max(1, n_epochs - warmup_epo))
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_epo])

best_file = os.path.join(OUT_DIR, f'{MODEL_NAME}_best.pth')
last_file = os.path.join(OUT_DIR, f'{MODEL_NAME}_last.pth')
h = {'tl': [], 'vl': [], 'qwk': [], 'qk': [], 'qr': []}
start_epoch, qwk_max = 1, -1.

if CKPT_RESUME and os.path.exists(CKPT_RESUME):          # continue a previous session
    ck = torch.load(CKPT_RESUME, map_location='cpu')
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['opt'])
    scheduler.load_state_dict(ck['sched']); start_epoch = ck['epoch'] + 1
    qwk_max = ck['qwk_max']; h = ck.get('hist', h)
    print(f'RESUMED from epoch {ck["epoch"]} | best QWK so far {qwk_max:.4f}', flush=True)

for epoch in range(start_epoch, n_epochs + 1):
    tl = train_epoch(train_loader, optimizer, epoch)
    vl, acc, qwk, qk, qr = val_epoch(valid_loader, df_valid)
    scheduler.step()
    for k, v in zip(h, [tl, vl, qwk, qk, qr]): h[k].append(v)
    live(h)
    print(f'epoch {epoch:2d} | train {tl:.4f} | val {vl:.4f} | acc {acc:.1f} | '
          f'QWK {qwk:.4f} (K {qk:.3f} / R {qr:.3f})', flush=True)
    # always save a full-state 'last' checkpoint so we can resume
    torch.save({'model': model.state_dict(), 'opt': optimizer.state_dict(),
                'sched': scheduler.state_dict(), 'epoch': epoch,
                'qwk_max': qwk_max, 'hist': h}, last_file)
    if qwk > qwk_max:
        qwk_max = qwk; torch.save(model.state_dict(), best_file)
        print('   saved best ->', best_file, flush=True)
print('BEST validation QWK:', round(qwk_max, 4), flush=True)

In [ ]:
model.load_state_dict(torch.load(best_file)); model.to(device)
P, T = val_epoch(valid_loader, df_valid, get_output=True)
cm = confusion_matrix(T, P, labels=list(range(6)))
fig, ax = plt.subplots(figsize=(5.5, 4.5)); im = ax.imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, int(v), ha='center', va='center', color='white' if v > cm.max()/2 else 'black')
ax.set_xticks(range(6)); ax.set_yticks(range(6)); ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'Validation (QWK={safe_qwk(P, T):.3f})'); fig.colorbar(im); plt.tight_layout(); plt.show()